In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Make summary plots from:
matched_all_2024_2025_m4shifts_combined_ratio90_clean.csv

Outputs (saved in 'm4shift_plots' next to the CSV):
  1) scatter_awe_vs_saber_m0p75.png        (AWE T vs SABER T)
  2) hist_deltaT_m0p75.png                 (histogram of Delta T)
  3) map_deltaT_m0p75_basemap_rect.png     (rectangular map, lat -60..60 only)
  4) doy_lat_deltaT_m0p75.png              (Delta T over DOY vs latitude, lat -60..60 only)

Also writes:
  - summary_overall.csv
  - summary_by_latband.csv

Notes
  - Uses Basemap for the map. Install if needed: pip install basemap basemap-data-hires
  - 2024 is leap year. For 2025, DOY within 2025 then +366.
"""

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from datetime import datetime
import warnings

# --------------------- user settings ---------------------
CSV_PATH = "matched_all_2024_2025_m4shifts_combined_ratio90_clean.csv"

# Column names
COL_AWE_T   = "awe_temp_avg_unfiltered"                      # AWE temperature (K)
COL_SABER_T = "m4_T_K_shift_m0p75"                           # SABER temperature (K)
COL_DELTA_T = "delta_temp_m4_unfiltered_K_shift_m0p75"       # SABER - AWE (K)
COL_LAT     = "saber_lat"
COL_LON     = "saber_lon"
COL_TIME    = "saber_time"                                    # ISO string

# Plot ranges
DELTA_T_MIN, DELTA_T_MAX = -25.0, 25.0
LAT_MIN, LAT_MAX = -60.0, 60.0

# --------------------- helpers ---------------------
def parse_iso_time_to_datetime(s):
    # Handles strings like 2024-01-03T01:49:39.347000
    try:
        return datetime.fromisoformat(str(s))
    except Exception:
        try:
            return pd.to_datetime(s).to_pydatetime()
        except Exception:
            return None

def compute_doy_across_2024_2025(dt):
    """Return day number since 2024-01-01 (1-based),
    with 2025 offset by +366 (2024 is leap year)."""
    if dt is None:
        return np.nan
    if dt.year == 2024:
        return float(dt.timetuple().tm_yday)  # 1..366
    if dt.year == 2025:
        return float(dt.timetuple().tm_yday + 366)  # 367..731
    base = datetime(2024, 1, 1)
    return float((dt - base).days + 1)

def clean_df(df):
    need = [COL_AWE_T, COL_SABER_T, COL_DELTA_T, COL_LAT, COL_LON, COL_TIME]
    for col in need:
        if col not in df.columns:
            raise KeyError(f"Missing column: {col}")
    df = df.copy()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=FutureWarning)
        dt_list = [parse_iso_time_to_datetime(x) for x in df[COL_TIME].values]
    df["__dt"]  = dt_list
    df["__doy"] = [compute_doy_across_2024_2025(d) for d in dt_list]
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=[COL_AWE_T, COL_SABER_T, COL_DELTA_T, COL_LAT, COL_LON, "__doy"])
    return df

def iqr(x):
    q75 = np.nanpercentile(x, 75)
    q25 = np.nanpercentile(x, 25)
    return q75 - q25

def within(x, thr):
    x = np.asarray(x, dtype=float)
    return np.mean(np.abs(x) <= thr) * 100.0

# --------------------- main ---------------------
def main():
    csv_path = Path(CSV_PATH)
    out_dir = csv_path.parent / "m4shift_plots"
    out_dir.mkdir(exist_ok=True, parents=True)

    df = pd.read_csv(csv_path)
    df = clean_df(df)

    # --------------------- statistics ---------------------
    n_total = len(df)
    mean_dt = np.nanmean(df[COL_DELTA_T])
    std_dt  = np.nanstd(df[COL_DELTA_T], ddof=1)
    med_dt  = np.nanmedian(df[COL_DELTA_T])
    iqr_dt  = iqr(df[COL_DELTA_T])
    min_dt  = np.nanmin(df[COL_DELTA_T])
    max_dt  = np.nanmax(df[COL_DELTA_T])
    rmse_dt = np.sqrt(np.nanmean((df[COL_DELTA_T])**2))
    mae_dt  = np.nanmean(np.abs(df[COL_DELTA_T]))

    r = np.corrcoef(df[COL_AWE_T], df[COL_SABER_T])[0,1]
    slope, intercept = np.polyfit(df[COL_AWE_T], df[COL_SABER_T], 1)

    p_within_5  = within(df[COL_DELTA_T], 5)
    p_within_10 = within(df[COL_DELTA_T], 10)
    p_within_15 = within(df[COL_DELTA_T], 15)

    print("\n=== Overall statistics ===")
    print(f"Rows: {n_total}")
    print(f"ΔT mean = {mean_dt:.2f} K, std = {std_dt:.2f} K, median = {med_dt:.2f} K, IQR = {iqr_dt:.2f} K")
    print(f"ΔT min = {min_dt:.2f} K, max = {max_dt:.2f} K")
    print(f"RMSE = {rmse_dt:.2f} K, MAE = {mae_dt:.2f} K")
    print(f"Corr(AWE, SABER) r = {r:.3f}")
    print(f"SABER ≈ {slope:.3f} * AWE + {intercept:.2f}")
    print("\nWithin thresholds:")
    print(f"|ΔT| ≤ 5 K:  {p_within_5:5.1f}%")
    print(f"|ΔT| ≤ 10 K: {p_within_10:5.1f}%")
    print(f"|ΔT| ≤ 15 K: {p_within_15:5.1f}%")

    bands = [(-60, -30), (-30, 0), (0, 30), (30, 60)]
    rows = []
    for lo, hi in bands:
        m = (df[COL_LAT] >= lo) & (df[COL_LAT] < hi)
        sub = df.loc[m, COL_DELTA_T]
        if sub.empty:
            rows.append([f"{lo}..{hi}", 0, np.nan, np.nan, np.nan, np.nan, np.nan])
        else:
            rows.append([
                f"{lo}..{hi}", sub.size,
                np.nanmean(sub), np.nanstd(sub, ddof=1),
                np.nanmedian(sub), iqr(sub), within(sub, 10)
            ])

    lat_stats_df = pd.DataFrame(rows, columns=[
        "lat_band_deg", "N", "mean_K", "std_K", "median_K", "IQR_K", "pct_within_10K"
    ])

    overall_df = pd.DataFrame({
        "N":[n_total],
        "mean_K":[mean_dt],
        "std_K":[std_dt],
        "median_K":[med_dt],
        "IQR_K":[iqr_dt],
        "min_K":[min_dt],
        "max_K":[max_dt],
        "RMSE_K":[rmse_dt],
        "MAE_K":[mae_dt],
        "corr_AWE_SABER":[r],
        "slope_SABER_vs_AWE":[slope],
        "intercept":[intercept],
        "pct_|dT|<=5K":[p_within_5],
        "pct_|dT|<=10K":[p_within_10],
        "pct_|dT|<=15K":[p_within_15],
    })

    overall_df.to_csv(out_dir / "summary_overall.csv", index=False)
    lat_stats_df.to_csv(out_dir / "summary_by_latband.csv", index=False)
    print(f"\nSaved summary CSVs to: {out_dir}")

    # --------------------- plots ---------------------
    aweT   = df[COL_AWE_T].values
    saberT = df[COL_SABER_T].values
    dT     = df[COL_DELTA_T].values
    lat    = df[COL_LAT].values
    lon    = df[COL_LON].values
    doy    = df["__doy"].values

    # 1) AWE T vs SABER T
    plt.figure(figsize=(6.5, 6.5))
    plt.scatter(aweT, saberT, s=6, alpha=0.5)
    plt.xlabel("AWE T (K)")
    plt.ylabel("SABER T (K)")
    mmin = float(np.nanmin([np.nanmin(aweT), np.nanmin(saberT)]))
    mmax = float(np.nanmax([np.nanmax(aweT), np.nanmax(saberT)]))
    pad  = 2.0
    lo, hi = mmin - pad, mmax + pad
    plt.plot([lo, hi], [lo, hi], lw=1, linestyle="--")
    plt.xlim(lo, hi)
    plt.ylim(lo, hi)
    plt.title("AWE T vs SABER T (m0.75)")
    plt.tight_layout()
    plt.savefig(out_dir / "scatter_awe_vs_saber_m0p75.png", dpi=200)
    plt.close()

    # 2) Histogram of Delta T
    plt.figure(figsize=(7.2, 4.6))
    bins = np.arange(DELTA_T_MIN, DELTA_T_MAX + 1, 1.0)
    plt.hist(
        np.clip(dT, DELTA_T_MIN, DELTA_T_MAX),
        bins=bins,
        edgecolor="black",       # add black borders
        linewidth=0.6,           # thin border line
        alpha=0.85,
        color="#2E86C1"          # clean blue tone
    )
    plt.axvline(0, color="k", linewidth=1.0, linestyle="--", alpha=0.8)  # zero line
    plt.xlim(DELTA_T_MIN, DELTA_T_MAX)
    plt.xlabel("Delta T (SABER - AWE) [K]")
    plt.ylabel("Count")
    plt.title("Delta T histogram (m0.75)")
    plt.grid(axis="y", linestyle="--", linewidth=0.4, alpha=0.7)
    plt.tight_layout()
    plt.savefig(out_dir / "hist_deltaT_m0p75.png", dpi=220)
    plt.close()


    # 3) Rectangular map with Basemap (lat -60..60)
    try:
        from mpl_toolkits.basemap import Basemap
        sel = (lat >= LAT_MIN) & (lat <= LAT_MAX) & np.isfinite(dT)
        lat_sel = lat[sel]
        lon_sel = lon[sel]
        dT_sel  = dT[sel]

        plt.figure(figsize=(10, 4.8))
        m = Basemap(projection="cyl",
                    llcrnrlat=LAT_MIN, urcrnrlat=LAT_MAX,
                    llcrnrlon=-180,    urcrnrlon=180,
                    resolution="c")

        m.drawcoastlines(linewidth=0.4)
        m.drawparallels(np.arange(-60, 61, 15), labels=[1,0,0,0], linewidth=0.3, dashes=[2,2])
        m.drawmeridians(np.arange(-180, 181, 30), labels=[0,0,0,1], linewidth=0.3, dashes=[2,2])
        m.drawmapboundary(fill_color="white")

        x, y = m(lon_sel, lat_sel)
        norm = TwoSlopeNorm(vmin=DELTA_T_MIN, vcenter=0.0, vmax=DELTA_T_MAX)
        sc = m.scatter(x, y, c=dT_sel, s=6, cmap="coolwarm", norm=norm, alpha=0.85, zorder=5)

        cb = plt.colorbar(sc, shrink=0.85, pad=0.03)
        cb.set_label("Delta T (K)")
        plt.title("Delta T (SABER - AWE), lat -60..60 (m0.75)")
        plt.tight_layout()
        plt.savefig(out_dir / "map_deltaT_m0p75_basemap_rect.png", dpi=220)
        plt.close()
    except Exception as e:
        print("Basemap plot skipped:", str(e))

    # 4) DOY vs Latitude, colored by Delta T (lat -60..60)
    sel2 = (lat >= LAT_MIN) & (lat <= LAT_MAX) & np.isfinite(dT) & np.isfinite(doy)
    plt.figure(figsize=(10, 5.4))
    norm = TwoSlopeNorm(vmin=DELTA_T_MIN, vcenter=0.0, vmax=DELTA_T_MAX)
    sc = plt.scatter(doy[sel2], lat[sel2], c=dT[sel2], s=6, cmap="coolwarm", norm=norm, alpha=0.85)
    plt.xlabel("Day since 2024-01-01")
    plt.ylabel("Latitude (deg)")
    plt.yticks(np.arange(-60, 61, 15))
    plt.ylim(LAT_MIN, LAT_MAX)
    cb = plt.colorbar(sc, pad=0.015)
    cb.set_label("Delta T (K)")
    plt.title("Delta T over DOY vs Latitude (lat -60..60, m0.75)")
    plt.tight_layout(rect=[0.04, 0.06, 0.98, 0.98])  # keep colorbar off x-label
    plt.savefig(out_dir / "doy_lat_deltaT_m0p75.png", dpi=220)
    plt.close()

    print(f"\nSaved plots to: {out_dir}")

if __name__ == "__main__":
    main()



=== Overall statistics ===
Rows: 13122
ΔT mean = -0.62 K, std = 5.01 K, median = -0.59 K, IQR = 6.10 K
ΔT min = -24.29 K, max = 48.57 K
RMSE = 5.04 K, MAE = 3.84 K
Corr(AWE, SABER) r = 0.906
SABER ≈ 0.774 * AWE + 43.78

Within thresholds:
|ΔT| ≤ 5 K:   71.3%
|ΔT| ≤ 10 K:  95.3%
|ΔT| ≤ 15 K:  99.3%

Saved summary CSVs to: m4shift_plots

Saved plots to: m4shift_plots


In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Jupyter script: summary + plots for three SABER methods (simple, m2, m3).

Input CSV:
  matched_all_2024_2025_methods_ratio90_clean.csv

Columns expected:
  AWE temp:        awe_temp_avg_unfiltered
  SABER temps:     saber_temp, m2_T_K, m3_T_K
  Delta Ts:        delta_temp_simple_unfiltered_K, delta_temp_m2_unfiltered_K, delta_temp_m3_unfiltered_K
  Coords/time:     saber_lat, saber_lon, saber_time (ISO)

Outputs (per method, saved next to CSV in 'methods_plots'):
  1) scatter_awe_vs_saber_<tag>.png
  2) hist_deltaT_<tag>.png
  3) map_deltaT_<tag>_basemap_rect.png       (lat -60..60 only; skipped if Basemap missing)
  4) doy_lat_deltaT_<tag>.png                 (lat -60..60 only)
  - summary_overall_<tag>.csv
  - summary_by_latband_<tag>.csv

Also writes:
  - summary_overall_all_methods.csv (one row per method)

Notes
  - 2024 is leap year. DOY in 2025 are offset by +366.
  - Basemap: pip install basemap basemap-data-hires (optional).
"""

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from datetime import datetime
import warnings

# --------------------- user settings ---------------------
CSV_PATH = "matched_all_2024_2025_methods_ratio90_clean.csv"

# Common columns
COL_AWE_T = "awe_temp_avg_unfiltered"   # AWE temperature (K)
COL_LAT   = "saber_lat"
COL_LON   = "saber_lon"
COL_TIME  = "saber_time"                # ISO string

# Methods to process: name -> columns + short tag used in filenames/titles
METHODS = {
    "simple": {"saber_col": "saber_temp", "delta_col": "delta_temp_simple_unfiltered_K", "tag": "simple"},
    "m2":     {"saber_col": "m2_T_K",     "delta_col": "delta_temp_m2_unfiltered_K",     "tag": "m2"},
    "m3":     {"saber_col": "m3_T_K",     "delta_col": "delta_temp_m3_unfiltered_K",     "tag": "m3"},
}

# Plot ranges
DELTA_T_MIN, DELTA_T_MAX = -25.0, 25.0
LAT_MIN, LAT_MAX = -60.0, 60.0

# --------------------- helpers ---------------------
def parse_iso_time_to_datetime(s):
    # Handles strings like 2024-01-03T01:49:39.347000
    try:
        return datetime.fromisoformat(str(s))
    except Exception:
        try:
            return pd.to_datetime(s).to_pydatetime()
        except Exception:
            return None

def compute_doy_across_2024_2025(dt):
    """Return day number since 2024-01-01 (1-based),
    with 2025 offset by +366 (2024 is leap year)."""
    if dt is None:
        return np.nan
    if dt.year == 2024:
        return float(dt.timetuple().tm_yday)  # 1..366
    if dt.year == 2025:
        return float(dt.timetuple().tm_yday + 366)  # 367..731
    base = datetime(2024, 1, 1)
    return float((dt - base).days + 1)

def clean_df(df, needed_cols):
    for col in needed_cols + [COL_AWE_T, COL_LAT, COL_LON, COL_TIME]:
        if col not in df.columns:
            raise KeyError(f"Missing column: {col}")
    df = df.copy()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=FutureWarning)
        dt_list = [parse_iso_time_to_datetime(x) for x in df[COL_TIME].values]
    df["__dt"]  = dt_list
    df["__doy"] = [compute_doy_across_2024_2025(d) for d in dt_list]
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=[COL_AWE_T, COL_LAT, COL_LON, "__doy"] + needed_cols)
    return df

def iqr(x):
    q75 = np.nanpercentile(x, 75)
    q25 = np.nanpercentile(x, 25)
    return q75 - q25

def within(x, thr):
    x = np.asarray(x, dtype=float)
    return np.mean(np.abs(x) <= thr) * 100.0

def method_stats_and_plots(df, saber_col, delta_col, tag, out_dir):
    aweT   = df[COL_AWE_T].values
    saberT = df[saber_col].values
    dT     = df[delta_col].values
    lat    = df[COL_LAT].values
    lon    = df[COL_LON].values
    doy    = df["__doy"].values

    # --------------------- statistics ---------------------
    n_total = len(df)
    mean_dt = np.nanmean(dT)
    std_dt  = np.nanstd(dT, ddof=1)
    med_dt  = np.nanmedian(dT)
    iqr_dt  = iqr(dT)
    min_dt  = np.nanmin(dT)
    max_dt  = np.nanmax(dT)
    rmse_dt = np.sqrt(np.nanmean((dT)**2))
    mae_dt  = np.nanmean(np.abs(dT))

    # Corr + linear fit
    try:
        r = np.corrcoef(aweT, saberT)[0, 1]
    except Exception:
        r = np.nan
    try:
        slope, intercept = np.polyfit(aweT, saberT, 1)
    except Exception:
        slope, intercept = np.nan, np.nan

    p_within_5  = within(dT, 5)
    p_within_10 = within(dT, 10)
    p_within_15 = within(dT, 15)

    # Lat bands
    bands = [(-60, -30), (-30, 0), (0, 30), (30, 60)]
    rows = []
    for lo, hi in bands:
        m = (lat >= lo) & (lat < hi)
        sub = dT[m]
        if sub.size == 0:
            rows.append([f"{lo}..{hi}", 0, np.nan, np.nan, np.nan, np.nan, np.nan])
        else:
            rows.append([
                f"{lo}..{hi}", sub.size,
                np.nanmean(sub), np.nanstd(sub, ddof=1),
                np.nanmedian(sub), iqr(sub), within(sub, 10)
            ])

    lat_stats_df = pd.DataFrame(rows, columns=[
        "lat_band_deg", "N", "mean_K", "std_K", "median_K", "IQR_K", "pct_within_10K"
    ])

    overall_df = pd.DataFrame({
        "method":[tag],
        "N":[n_total],
        "mean_K":[mean_dt],
        "std_K":[std_dt],
        "median_K":[med_dt],
        "IQR_K":[iqr_dt],
        "min_K":[min_dt],
        "max_K":[max_dt],
        "RMSE_K":[rmse_dt],
        "MAE_K":[mae_dt],
        "corr_AWE_SABER":[r],
        "slope_SABER_vs_AWE":[slope],
        "intercept":[intercept],
        "pct_|dT|<=5K":[p_within_5],
        "pct_|dT|<=10K":[p_within_10],
        "pct_|dT|<=15K":[p_within_15],
    })

    # Write summaries
    overall_df.to_csv(out_dir / f"summary_overall_{tag}.csv", index=False)
    lat_stats_df.to_csv(out_dir / f"summary_by_latband_{tag}.csv", index=False)

    # --------------------- plots ---------------------
    # 1) AWE T vs SABER T
    plt.figure(figsize=(6.5, 6.5))
    plt.scatter(aweT, saberT, s=6, alpha=0.5)
    plt.xlabel("AWE T (K)")
    plt.ylabel(f"SABER T (K) [{tag}]")
    mmin = float(np.nanmin([np.nanmin(aweT), np.nanmin(saberT)]))
    mmax = float(np.nanmax([np.nanmax(aweT), np.nanmax(saberT)]))
    pad  = 2.0
    lo, hi = mmin - pad, mmax + pad
    plt.plot([lo, hi], [lo, hi], lw=1, linestyle="--")
    plt.xlim(lo, hi)
    plt.ylim(lo, hi)
    plt.title(f"AWE T vs SABER T ({tag})")
    plt.tight_layout()
    plt.savefig(out_dir / f"scatter_awe_vs_saber_{tag}.png", dpi=200)
    plt.close()

    # 2) Histogram of Delta T
    plt.figure(figsize=(7.2, 4.6))
    bins = np.arange(DELTA_T_MIN, DELTA_T_MAX + 1, 1.0)
    plt.hist(
        np.clip(dT, DELTA_T_MIN, DELTA_T_MAX),
        bins=bins,
        edgecolor="black",
        linewidth=0.6,
        alpha=0.85,
    )
    plt.axvline(0, color="k", linewidth=1.0, linestyle="--", alpha=0.8)  # zero line
    plt.xlim(DELTA_T_MIN, DELTA_T_MAX)
    plt.xlabel("Delta T (SABER - AWE) [K]")
    plt.ylabel("Count")
    plt.title(f"Delta T histogram ({tag})")
    plt.grid(axis="y", linestyle="--", linewidth=0.4, alpha=0.7)
    plt.tight_layout()
    plt.savefig(out_dir / f"hist_deltaT_{tag}.png", dpi=220)
    plt.close()

    # 3) Rectangular map with Basemap (lat -60..60)
    try:
        from mpl_toolkits.basemap import Basemap
        sel = (lat >= LAT_MIN) & (lat <= LAT_MAX) & np.isfinite(dT)
        lat_sel = lat[sel]
        lon_sel = lon[sel]
        dT_sel  = dT[sel]

        plt.figure(figsize=(10, 4.8))
        m = Basemap(projection="cyl",
                    llcrnrlat=LAT_MIN, urcrnrlat=LAT_MAX,
                    llcrnrlon=-180,    urcrnrlon=180,
                    resolution="c")

        m.drawcoastlines(linewidth=0.4)
        m.drawparallels(np.arange(-60, 61, 15), labels=[1,0,0,0], linewidth=0.3, dashes=[2,2])
        m.drawmeridians(np.arange(-180, 181, 30), labels=[0,0,0,1], linewidth=0.3, dashes=[2,2])
        m.drawmapboundary(fill_color="white")

        x, y = m(lon_sel, lat_sel)
        norm = TwoSlopeNorm(vmin=DELTA_T_MIN, vcenter=0.0, vmax=DELTA_T_MAX)
        sc = m.scatter(x, y, c=dT_sel, s=6, cmap="coolwarm", norm=norm, alpha=0.85, zorder=5)

        cb = plt.colorbar(sc, shrink=0.85, pad=0.03)
        cb.set_label("Delta T (K)")
        plt.title(f"Delta T (SABER - AWE), lat -60..60 ({tag})")
        plt.tight_layout()
        plt.savefig(out_dir / f"map_deltaT_{tag}_basemap_rect.png", dpi=220)
        plt.close()
    except Exception as e:
        print(f"Basemap plot skipped for {tag}:", str(e))

    # 4) DOY vs Latitude, colored by Delta T (lat -60..60)
    sel2 = (lat >= LAT_MIN) & (lat <= LAT_MAX) & np.isfinite(dT) & np.isfinite(doy)
    plt.figure(figsize=(10, 5.4))
    norm = TwoSlopeNorm(vmin=DELTA_T_MIN, vcenter=0.0, vmax=DELTA_T_MAX)
    sc = plt.scatter(doy[sel2], lat[sel2], c=dT[sel2], s=6, cmap="coolwarm", norm=norm, alpha=0.85)
    plt.xlabel("Day since 2024-01-01")
    plt.ylabel("Latitude (deg)")
    plt.yticks(np.arange(-60, 61, 15))
    plt.ylim(LAT_MIN, LAT_MAX)
    cb = plt.colorbar(sc, pad=0.015)
    cb.set_label("Delta T (K)")
    plt.title(f"Delta T over DOY vs Latitude (lat -60..60, {tag})")
    plt.tight_layout(rect=[0.04, 0.06, 0.98, 0.98])
    plt.savefig(out_dir / f"doy_lat_deltaT_{tag}.png", dpi=220)
    plt.close()

    return overall_df

# --------------------- main (safe for Jupyter) ---------------------
def run_all_methods():
    csv_path = Path(CSV_PATH)
    out_dir = csv_path.parent / "methods_plots"
    out_dir.mkdir(exist_ok=True, parents=True)

    df_raw = pd.read_csv(csv_path)

    overall_rows = []
    for name, spec in METHODS.items():
        saber_col = spec["saber_col"]
        delta_col = spec["delta_col"]
        tag       = spec["tag"]

        # Clean specifically for the required columns
        df = clean_df(df_raw, needed_cols=[saber_col, delta_col])

        print(f"\n=== {name.upper()} ({tag}) ===")
        res_df = method_stats_and_plots(df, saber_col, delta_col, tag, out_dir)
        print(res_df.to_string(index=False))
        overall_rows.append(res_df)

    # Combined summary across methods
    overall_all = pd.concat(overall_rows, ignore_index=True)
    overall_all.to_csv(out_dir / "summary_overall_all_methods.csv", index=False)
    print(f"\nSaved all outputs to: {out_dir}")

# If running as a script cell in Jupyter, just call:
run_all_methods()



=== SIMPLE (simple) ===
method     N    mean_K    std_K  median_K     IQR_K      min_K     max_K   RMSE_K    MAE_K  corr_AWE_SABER  slope_SABER_vs_AWE  intercept  pct_|dT|<=5K  pct_|dT|<=10K  pct_|dT|<=15K
simple 13122 -0.696379 8.983066 -0.406977 11.297697 -45.052121 50.653098 9.009676 6.976055        0.784554            0.967525   5.686262     44.894071      75.796373      90.885536

=== M2 (m2) ===
method     N    mean_K    std_K  median_K    IQR_K      min_K     max_K   RMSE_K    MAE_K  corr_AWE_SABER  slope_SABER_vs_AWE  intercept  pct_|dT|<=5K  pct_|dT|<=10K  pct_|dT|<=15K
    m2 13122 -1.083112 5.073661 -0.997727 6.157267 -23.530856 52.342935 5.187794 3.951264        0.904609            0.753206  47.421798     70.278921      94.810242      99.260783

=== M3 (m3) ===
method     N    mean_K   std_K  median_K    IQR_K      min_K     max_K   RMSE_K    MAE_K  corr_AWE_SABER  slope_SABER_vs_AWE  intercept  pct_|dT|<=5K  pct_|dT|<=10K  pct_|dT|<=15K
    m3 13122 -0.007165 5.07357 -0.0

In [7]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Jupyter script: summary + plots for SABER Methods 1–4
Now saves ALL plots to a single folder and adds 1×4 multi-panel figures.

Inputs:
  - Methods 1–3 CSV: matched_all_2024_2025_methods_ratio90_clean.csv
  - Method 4 CSV:    matched_all_2024_2025_m4shifts_combined_ratio90_clean.csv

Expected columns
  Common:
    AWE temp:  awe_temp_avg_unfiltered
    Coords:    saber_lat, saber_lon
    Time:      saber_time (ISO)
  Method 1 (simple):
    saber_temp, delta_temp_simple_unfiltered_K
  Method 2 (m2):
    m2_T_K,    delta_temp_m2_unfiltered_K
  Method 3 (m3):
    m3_T_K,    delta_temp_m3_unfiltered_K
  Method 4 (m4, shift = -0.75 km):
    m4_T_K_shift_m0p75, delta_temp_m4_unfiltered_K_shift_m0p75

Outputs (all saved to ONE folder: <CSV_123_parent>/all_methods_plots)
  Per method:
    - scatter_awe_vs_saber_<tag>.png
    - hist_deltaT_<tag>.png
    - map_deltaT_<tag>_basemap_rect.png   (lat -60..60 only; auto-skip if Basemap missing)
    - doy_lat_deltaT_<tag>.png            (lat -60..60 only)
    - summary_overall_<tag>.csv
    - summary_by_latband_<tag>.csv
  Combined:
    - summary_overall_all_methods.csv
    - scatter_all_methods.png             (1×4)
    - hist_all_methods.png                (1×4)

Notes
  - 2024 is leap year. DOY in 2025 are offset by +366.
  - Basemap (optional): pip install basemap basemap-data-hires
"""

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from datetime import datetime
import warnings

# --------------------- user settings ---------------------
CSV_PATH_123 = "matched_all_2024_2025_methods_ratio90_clean.csv"
CSV_PATH_M4  = "matched_all_2024_2025_m4shifts_combined_ratio90_clean.csv"

# Common columns
COL_AWE_T = "awe_temp_avg_unfiltered"
COL_LAT   = "saber_lat"
COL_LON   = "saber_lon"
COL_TIME  = "saber_time"  # ISO string

# Plot ranges
DELTA_T_MIN, DELTA_T_MAX = -25.0, 25.0
LAT_MIN, LAT_MAX = -60.0, 60.0

# Methods: Method name -> (which_csv, saber_col, delta_col, short tag)
METHODS = {
    "Method 1": ("123", "saber_temp",                    "delta_temp_simple_unfiltered_K",           "m1"),
    "Method 2": ("123", "m2_T_K",                        "delta_temp_m2_unfiltered_K",               "m2"),
    "Method 3": ("123", "m3_T_K",                        "delta_temp_m3_unfiltered_K",               "m3"),
    "Method 4": ("m4",  "m4_T_K_shift_m0p75",            "delta_temp_m4_unfiltered_K_shift_m0p75",   "m4_m0p75"),
}

# --------------------- helpers ---------------------
def parse_iso_time_to_datetime(s):
    try:
        return datetime.fromisoformat(str(s))
    except Exception:
        try:
            return pd.to_datetime(s).to_pydatetime()
        except Exception:
            return None

def compute_doy_across_2024_2025(dt):
    """Return day number since 2024-01-01 (1-based),
    with 2025 offset by +366 (2024 is leap year)."""
    if dt is None:
        return np.nan
    if dt.year == 2024:
        return float(dt.timetuple().tm_yday)  # 1..366
    if dt.year == 2025:
        return float(dt.timetuple().tm_yday + 366)  # 367..731
    base = datetime(2024, 1, 1)
    return float((dt - base).days + 1)

def clean_df(df, needed_cols):
    for col in needed_cols + [COL_AWE_T, COL_LAT, COL_LON, COL_TIME]:
        if col not in df.columns:
            raise KeyError(f"Missing column: {col}")
    df = df.copy()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=FutureWarning)
        dt_list = [parse_iso_time_to_datetime(x) for x in df[COL_TIME].values]
    df["__dt"]  = dt_list
    df["__doy"] = [compute_doy_across_2024_2025(d) for d in dt_list]
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=[COL_AWE_T, COL_LAT, COL_LON, "__doy"] + needed_cols)
    return df

def iqr(x):
    q75 = np.nanpercentile(x, 75)
    q25 = np.nanpercentile(x, 25)
    return q75 - q25

def within(x, thr):
    x = np.asarray(x, dtype=float)
    return np.mean(np.abs(x) <= thr) * 100.0

def method_stats_and_plots(df, method_name, saber_col, delta_col, tag, out_dir):
    aweT   = df[COL_AWE_T].values
    saberT = df[saber_col].values
    dT     = df[delta_col].values
    lat    = df[COL_LAT].values
    lon    = df[COL_LON].values
    doy    = df["__doy"].values

    # --------------------- statistics ---------------------
    n_total = len(df)
    mean_dt = np.nanmean(dT)
    std_dt  = np.nanstd(dT, ddof=1)
    med_dt  = np.nanmedian(dT)
    iqr_dt  = iqr(dT)
    min_dt  = np.nanmin(dT)
    max_dt  = np.nanmax(dT)
    rmse_dt = np.sqrt(np.nanmean((dT)**2))
    mae_dt  = np.nanmean(np.abs(dT))

    try:
        r = np.corrcoef(aweT, saberT)[0, 1]
    except Exception:
        r = np.nan
    try:
        slope, intercept = np.polyfit(aweT, saberT, 1)
    except Exception:
        slope, intercept = np.nan, np.nan

    p_within_5  = within(dT, 5)
    p_within_10 = within(dT, 10)
    p_within_15 = within(dT, 15)

    # Lat bands
    bands = [(-60, -30), (-30, 0), (0, 30), (30, 60)]
    rows = []
    for lo, hi in bands:
        m = (lat >= lo) & (lat < hi)
        sub = dT[m]
        if sub.size == 0:
            rows.append([f"{lo}..{hi}", 0, np.nan, np.nan, np.nan, np.nan, np.nan])
        else:
            rows.append([
                f"{lo}..{hi}", sub.size,
                np.nanmean(sub), np.nanstd(sub, ddof=1),
                np.nanmedian(sub), iqr(sub), within(sub, 10)
            ])

    lat_stats_df = pd.DataFrame(rows, columns=[
        "lat_band_deg", "N", "mean_K", "std_K", "median_K", "IQR_K", "pct_within_10K"
    ])

    overall_df = pd.DataFrame({
        "method":[method_name],
        "tag":[tag],
        "N":[n_total],
        "mean_K":[mean_dt],
        "std_K":[std_dt],
        "median_K":[med_dt],
        "IQR_K":[iqr_dt],
        "min_K":[min_dt],
        "max_K":[max_dt],
        "RMSE_K":[rmse_dt],
        "MAE_K":[mae_dt],
        "corr_AWE_SABER":[r],
        "slope_SABER_vs_AWE":[slope],
        "intercept":[intercept],
        "pct_|dT|<=5K":[p_within_5],
        "pct_|dT|<=10K":[p_within_10],
        "pct_|dT|<=15K":[p_within_15],
    })

    # Write summaries
    overall_df.to_csv(out_dir / f"summary_overall_{tag}.csv", index=False)
    lat_stats_df.to_csv(out_dir / f"summary_by_latband_{tag}.csv", index=False)

    # --------------------- per-method plots ---------------------
    # 1) AWE T vs SABER T
    plt.figure(figsize=(6.4, 6.4))
    plt.scatter(aweT, saberT, s=6, alpha=0.5)
    plt.xlabel("AWE T (K)")
    plt.ylabel(f"SABER T (K) [{method_name}]")
    mmin = float(np.nanmin([np.nanmin(aweT), np.nanmin(saberT)]))
    mmax = float(np.nanmax([np.nanmax(aweT), np.nanmax(saberT)]))
    pad  = 2.0
    lo, hi = mmin - pad, mmax + pad
    plt.plot([lo, hi], [lo, hi], lw=1, linestyle="--")
    plt.xlim(lo, hi)
    plt.ylim(lo, hi)
    plt.title(f"AWE T vs SABER T ({method_name})")
    plt.tight_layout()
    plt.savefig(out_dir / f"scatter_awe_vs_saber_{tag}.png", dpi=200)
    plt.close()

    # 2) Histogram of Delta T
    plt.figure(figsize=(7.2, 4.6))
    bins = np.arange(DELTA_T_MIN, DELTA_T_MAX + 1, 1.0)
    plt.hist(
        np.clip(dT, DELTA_T_MIN, DELTA_T_MAX),
        bins=bins,
        edgecolor="black",
        linewidth=0.6,
        alpha=0.88,
    )
    plt.axvline(0, color="k", linewidth=1.0, linestyle="--", alpha=0.85)
    plt.xlim(DELTA_T_MIN, DELTA_T_MAX)
    plt.xlabel("Delta T (SABER - AWE) [K]")
    plt.ylabel("Count")
    plt.title(f"Delta T histogram ({method_name})")
    plt.grid(axis="y", linestyle="--", linewidth=0.4, alpha=0.7)
    plt.tight_layout()
    plt.savefig(out_dir / f"hist_deltaT_{tag}.png", dpi=220)
    plt.close()

    # 3) Rectangular map with Basemap (lat -60..60)
    try:
        from mpl_toolkits.basemap import Basemap
        sel = (lat >= LAT_MIN) & (lat <= LAT_MAX) & np.isfinite(dT)
        lat_sel = lat[sel]
        lon_sel = lon[sel]
        dT_sel  = dT[sel]

        plt.figure(figsize=(10, 4.8))
        m = Basemap(projection="cyl",
                    llcrnrlat=LAT_MIN, urcrnrlat=LAT_MAX,
                    llcrnrlon=-180,    urcrnrlon=180,
                    resolution="c")

        m.drawcoastlines(linewidth=0.4)
        m.drawparallels(np.arange(-60, 61, 15), labels=[1,0,0,0], linewidth=0.3, dashes=[2,2])
        m.drawmeridians(np.arange(-180, 181, 30), labels=[0,0,0,1], linewidth=0.3, dashes=[2,2])
        m.drawmapboundary(fill_color="white")

        x, y = m(lon_sel, lat_sel)
        norm = TwoSlopeNorm(vmin=DELTA_T_MIN, vcenter=0.0, vmax=DELTA_T_MAX)
        sc = m.scatter(x, y, c=dT_sel, s=6, cmap="coolwarm", norm=norm, alpha=0.85, zorder=5)

        cb = plt.colorbar(sc, shrink=0.85, pad=0.03)
        cb.set_label("Delta T (K)")
        plt.title(f"Delta T (SABER - AWE), lat -60..60 ({method_name})")
        plt.tight_layout()
        plt.savefig(out_dir / f"map_deltaT_{tag}_basemap_rect.png", dpi=220)
        plt.close()
    except Exception as e:
        print(f"Basemap plot skipped for {method_name}:", str(e))

    # 4) DOY vs Latitude, colored by Delta T (lat -60..60)
    sel2 = (lat >= LAT_MIN) & (lat <= LAT_MAX) & np.isfinite(dT) & np.isfinite(doy)
    plt.figure(figsize=(10, 5.4))
    norm = TwoSlopeNorm(vmin=DELTA_T_MIN, vcenter=0.0, vmax=DELTA_T_MAX)
    sc = plt.scatter(doy[sel2], lat[sel2], c=dT[sel2], s=6, cmap="coolwarm", norm=norm, alpha=0.85)
    plt.xlabel("Day since 2024-01-01")
    plt.ylabel("Latitude (deg)")
    plt.yticks(np.arange(-60, 61, 15))
    plt.ylim(LAT_MIN, LAT_MAX)
    cb = plt.colorbar(sc, pad=0.015)
    cb.set_label("Delta T (K)")
    plt.title(f"Delta T over DOY vs Latitude (lat -60..60, {method_name})")
    plt.tight_layout(rect=[0.04, 0.06, 0.98, 0.98])
    plt.savefig(out_dir / f"doy_lat_deltaT_{tag}.png", dpi=220)
    plt.close()

    return overall_df, (aweT, saberT, dT)  # for combined plots

# --------------------- combined multipanel plots ---------------------
def make_combined_scatter(awe_saber_dict, out_dir):
    """
    awe_saber_dict: {method_name: (aweT, saberT)}
    Creates 1×4 scatter panel with common axes and 1:1 line.
    """
    methods_order = ["Method 1", "Method 2", "Method 3", "Method 4"]
    fig, axes = plt.subplots(1, 4, figsize=(18, 4.6), constrained_layout=True)

    # Common axis limits across all methods
    mins, maxs = [], []
    for name in methods_order:
        aweT, saberT = awe_saber_dict[name]
        mins.append(np.nanmin([aweT, saberT]))
        maxs.append(np.nanmax([aweT, saberT]))
    lo = float(np.nanmin(mins)) - 2.0
    hi = float(np.nanmax(maxs)) + 2.0

    for ax, name in zip(axes, methods_order):
        aweT, saberT = awe_saber_dict[name]
        ax.scatter(aweT, saberT, s=5, alpha=0.5)
        ax.plot([lo, hi], [lo, hi], linestyle="--", linewidth=1.0)
        ax.set_xlim(lo, hi)
        ax.set_ylim(lo, hi)
        ax.set_title(name)
        ax.set_xlabel("AWE T (K)")
        if ax is axes[0]:
            ax.set_ylabel("SABER T (K)")
        ax.grid(True, linestyle=":", linewidth=0.5, alpha=0.7)

    fig.suptitle("AWE T vs SABER T — Methods 1–4", y=1.02, fontsize=14)
    fig.savefig(out_dir / "scatter_all_methods.png", dpi=220, bbox_inches="tight")
    plt.close(fig)

def make_combined_hist(dT_dict, out_dir):
    """
    dT_dict: {method_name: dT_array}
    Creates 1×4 histogram panel with common bins/limits and zero line.
    """
    methods_order = ["Method 1", "Method 2", "Method 3", "Method 4"]
    fig, axes = plt.subplots(1, 4, figsize=(18, 4.6), constrained_layout=True)
    bins = np.arange(DELTA_T_MIN, DELTA_T_MAX + 1, 1.0)

    for ax, name in zip(axes, methods_order):
        dT = dT_dict[name]
        ax.hist(np.clip(dT, DELTA_T_MIN, DELTA_T_MAX), bins=bins,
                edgecolor="black", linewidth=0.6, alpha=0.88)
        ax.axvline(0, color="k", linewidth=1.0, linestyle="--", alpha=0.85)
        ax.set_xlim(DELTA_T_MIN, DELTA_T_MAX)
        ax.set_title(name)
        ax.set_xlabel("ΔT (SABER - AWE) [K]")
        if ax is axes[0]:
            ax.set_ylabel("Count")
        ax.grid(axis="y", linestyle="--", linewidth=0.4, alpha=0.7)

    fig.suptitle("ΔT Histograms — Methods 1–4", y=1.02, fontsize=14)
    fig.savefig(out_dir / "hist_all_methods.png", dpi=220, bbox_inches="tight")
    plt.close(fig)

# --------------------- main (safe for Jupyter) ---------------------
def run_all_methods():
    # Load both CSVs
    csv_path_123 = Path(CSV_PATH_123)
    csv_path_m4  = Path(CSV_PATH_M4)

    df123 = pd.read_csv(csv_path_123)
    dfm4  = pd.read_csv(csv_path_m4)

    # Single output folder
    out_dir_all = csv_path_123.parent / "all_methods_plots"
    out_dir_all.mkdir(exist_ok=True, parents=True)

    overall_rows = []
    awe_saber_for_combined = {}  # method -> (aweT, saberT)
    dT_for_combined = {}         # method -> dT

    for method_name, (which_csv, saber_col, delta_col, tag) in METHODS.items():
        df_raw = df123 if which_csv == "123" else dfm4

        # Clean specifically for required columns
        df = clean_df(df_raw, needed_cols=[saber_col, delta_col])

        print(f"\n=== {method_name} ({tag}) ===")
        res_df, arrays = method_stats_and_plots(df, method_name, saber_col, delta_col, tag, out_dir_all)
        print(res_df.to_string(index=False))
        overall_rows.append(res_df)

        aweT, saberT, dT = arrays
        awe_saber_for_combined[method_name] = (aweT, saberT)
        dT_for_combined[method_name] = dT

    # Combined summary (saved to the same single folder)
    overall_all = pd.concat(overall_rows, ignore_index=True)
    overall_all.to_csv(out_dir_all / "summary_overall_all_methods.csv", index=False)

    # Multi-panel combined figures
    make_combined_scatter({k: (v[0], v[1]) for k, v in awe_saber_for_combined.items()}, out_dir_all)
    make_combined_hist(dT_for_combined, out_dir_all)

    print(f"\nSaved all outputs to: {out_dir_all}")

# If running as a script cell in Jupyter, just call:
run_all_methods()



=== Method 1 (m1) ===
  method tag     N    mean_K    std_K  median_K     IQR_K      min_K     max_K   RMSE_K    MAE_K  corr_AWE_SABER  slope_SABER_vs_AWE  intercept  pct_|dT|<=5K  pct_|dT|<=10K  pct_|dT|<=15K
Method 1  m1 13122 -0.696379 8.983066 -0.406977 11.297697 -45.052121 50.653098 9.009676 6.976055        0.784554            0.967525   5.686262     44.894071      75.796373      90.885536

=== Method 2 (m2) ===
  method tag     N    mean_K    std_K  median_K    IQR_K      min_K     max_K   RMSE_K    MAE_K  corr_AWE_SABER  slope_SABER_vs_AWE  intercept  pct_|dT|<=5K  pct_|dT|<=10K  pct_|dT|<=15K
Method 2  m2 13122 -1.083112 5.073661 -0.997727 6.157267 -23.530856 52.342935 5.187794 3.951264        0.904609            0.753206  47.421798     70.278921      94.810242      99.260783

=== Method 3 (m3) ===
  method tag     N    mean_K   std_K  median_K    IQR_K      min_K     max_K   RMSE_K    MAE_K  corr_AWE_SABER  slope_SABER_vs_AWE  intercept  pct_|dT|<=5K  pct_|dT|<=10K  pct_|dT|<

In [53]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Summary + plots for SABER Methods 1–4 (paper version)
All plots saved to one folder, larger text for visibility on paper.
"""

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from datetime import datetime
import warnings

# --------------------- user settings ---------------------
CSV_PATH_123 = "matched_all_2024_2025_methods_ratio90_clean.csv"
CSV_PATH_M4  = "matched_all_2024_2025_m4shifts_combined_ratio90_clean.csv"

# Common columns
COL_AWE_T = "awe_temp_avg_unfiltered"
COL_LAT   = "saber_lat"
COL_LON   = "saber_lon"
COL_TIME  = "saber_time"

# Plot ranges
DELTA_T_MIN, DELTA_T_MAX = -25.0, 25.0
LAT_MIN, LAT_MAX = -60.0, 60.0

# Methods
METHODS = {
    "Method 1": ("123", "saber_temp",                    "delta_temp_simple_unfiltered_K",           "m1"),
    "Method 2": ("123", "m2_T_K",                        "delta_temp_m2_unfiltered_K",               "m2"),
    "Method 3": ("123", "m3_T_K",                        "delta_temp_m3_unfiltered_K",               "m3"),
    "Method 4": ("m4",  "m4_T_K_shift_m0p75",            "delta_temp_m4_unfiltered_K_shift_m0p75",   "m4_m0p75"),
}

# --------------------- style for paper ---------------------
plt.rcParams.update({
    "font.size": 19,
    "axes.labelsize": 20,
    "axes.titlesize": 21,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 18,
    "figure.titlesize": 23,
    "lines.linewidth": 1.8,
})

# --------------------- helpers ---------------------
def parse_iso_time_to_datetime(s):
    try:
        return datetime.fromisoformat(str(s))
    except Exception:
        try:
            return pd.to_datetime(s).to_pydatetime()
        except Exception:
            return None

def compute_doy_across_2024_2025(dt):
    if dt is None:
        return np.nan
    if dt.year == 2024:
        return float(dt.timetuple().tm_yday)
    if dt.year == 2025:
        return float(dt.timetuple().tm_yday + 366)
    base = datetime(2024, 1, 1)
    return float((dt - base).days + 1)

def clean_df(df, needed_cols):
    for col in needed_cols + [COL_AWE_T, COL_LAT, COL_LON, COL_TIME]:
        if col not in df.columns:
            raise KeyError(f"Missing column: {col}")
    df = df.copy()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=FutureWarning)
        dt_list = [parse_iso_time_to_datetime(x) for x in df[COL_TIME].values]
    df["__dt"]  = dt_list
    df["__doy"] = [compute_doy_across_2024_2025(d) for d in dt_list]
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=[COL_AWE_T, COL_LAT, COL_LON, "__doy"] + needed_cols)
    return df

def iqr(x):
    q75 = np.nanpercentile(x, 75)
    q25 = np.nanpercentile(x, 25)
    return q75 - q25

def within(x, thr):
    x = np.asarray(x, dtype=float)
    return np.mean(np.abs(x) <= thr) * 100.0

# --------------------- main plotting helper ---------------------
def method_stats_and_plots(df, method_name, saber_col, delta_col, tag, out_dir):
    aweT   = df[COL_AWE_T].values
    saberT = df[saber_col].values
    dT     = df[delta_col].values
    lat    = df[COL_LAT].values
    lon    = df[COL_LON].values
    doy    = df["__doy"].values

    # Statistics
    n_total = len(df)
    mean_dt = np.nanmean(dT)
    std_dt  = np.nanstd(dT, ddof=1)
    med_dt  = np.nanmedian(dT)
    iqr_dt  = iqr(dT)
    rmse_dt = np.sqrt(np.nanmean((dT)**2))
    mae_dt  = np.nanmean(np.abs(dT))

    # Corr + fit
    r = np.corrcoef(aweT, saberT)[0, 1]
    slope, intercept = np.polyfit(aweT, saberT, 1)

    # ----- per-method plots -----
    # 1) Scatter AWE vs SABER
    plt.figure(figsize=(7.2, 7.2))
    plt.scatter(aweT, saberT, s=10, alpha=0.55)
    plt.xlabel("AWE T (K)")
    plt.ylabel(f"SABER T (K)\n{method_name}")
    lo = min(aweT.min(), saberT.min()) - 2
    hi = max(aweT.max(), saberT.max()) + 2
    plt.plot([lo, hi], [lo, hi], "--", color="gray", linewidth=1.3)
    plt.xlim(lo, hi)
    plt.ylim(lo, hi)
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.savefig(out_dir / f"scatter_awe_vs_saber_{tag}.png", dpi=300)
    plt.close()

    # 2) Histogram
    plt.figure(figsize=(8, 5))
    bins = np.arange(DELTA_T_MIN, DELTA_T_MAX + 1, 1.0)
    plt.hist(np.clip(dT, DELTA_T_MIN, DELTA_T_MAX),
             bins=bins, edgecolor="black", linewidth=0.8, alpha=0.85)
    plt.axvline(0, color="k", linestyle="--", linewidth=1.2)
    plt.xlabel("ΔT (SABER - AWE) [K]")
    plt.ylabel("Count")
    plt.title(f"ΔT Distribution — {method_name}")
    plt.grid(axis="y", linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.savefig(out_dir / f"hist_deltaT_{tag}.png", dpi=300)
    plt.close()

    # 3) Map (optional Basemap)
    try:
        from mpl_toolkits.basemap import Basemap
        sel = (lat >= LAT_MIN) & (lat <= LAT_MAX) & np.isfinite(dT)
        plt.figure(figsize=(12, 5.2))
        m = Basemap(projection="cyl",
                    llcrnrlat=LAT_MIN, urcrnrlat=LAT_MAX,
                    llcrnrlon=-180, urcrnrlon=180, resolution="c")
        m.drawcoastlines(linewidth=0.6)
        m.drawparallels(np.arange(-60, 61, 15), labels=[1,0,0,0], fontsize=13)
        m.drawmeridians(np.arange(-180, 181, 30), labels=[0,0,0,1], fontsize=13)
        norm = TwoSlopeNorm(vmin=DELTA_T_MIN, vcenter=0, vmax=DELTA_T_MAX)
        sc = m.scatter(lon[sel], lat[sel], c=dT[sel], s=8, cmap="coolwarm", norm=norm, alpha=0.85)
        cb = plt.colorbar(sc, shrink=0.85, pad=0.03)
        cb.set_label("ΔT (K)", fontsize=15)
        plt.title(f"ΔT Map (lat -60..60) — {method_name}", fontsize=18)
        plt.tight_layout()
        plt.savefig(out_dir / f"map_deltaT_{tag}_basemap_rect.png", dpi=300)
        plt.close()
    except Exception:
        pass

    # 4) DOY vs Latitude
    sel2 = (lat >= LAT_MIN) & (lat <= LAT_MAX) & np.isfinite(dT)
    plt.figure(figsize=(11, 5.5))
    norm = TwoSlopeNorm(vmin=DELTA_T_MIN, vcenter=0.0, vmax=DELTA_T_MAX)
    sc = plt.scatter(doy[sel2], lat[sel2], c=dT[sel2], s=8, cmap="coolwarm", norm=norm, alpha=0.85)
    plt.xlabel("Day since 2024-01-01")
    plt.ylabel("Latitude (°)")
    plt.colorbar(sc, pad=0.015, label="ΔT (K)")
    plt.title(f"ΔT vs DOY & Latitude — {method_name}", fontsize=18)
    plt.tight_layout()
    plt.savefig(out_dir / f"doy_lat_deltaT_{tag}.png", dpi=300)
    plt.close()

    return (aweT, saberT, dT)


# --------------------- combined 1×4 plots ---------------------
def make_combined_scatter(awe_saber_dict, out_dir):
    methods = ["Method 1", "Method 2", "Method 3", "Method 4"]
    # Create shared x/y axes
    fig, axs = plt.subplots(1, 4, figsize=(17, 4.5),
                            constrained_layout=True, sharex=True, sharey=True)

    # Determine global min/max across all methods
    mins, maxs = [], []
    for name in methods:
        aweT, saberT = awe_saber_dict[name]
        mins.append(np.nanmin([aweT, saberT]))
        maxs.append(np.nanmax([aweT, saberT]))
    lo, hi = float(min(mins)) - 2, float(max(maxs)) + 2

    # Plot each scatter using the same limits
    for ax, name in zip(axs, methods):
        aweT, saberT = awe_saber_dict[name]
        ax.scatter(aweT, saberT, s=10, alpha=0.55)
        ax.plot([lo, hi], [lo, hi], "--", color="gray", linewidth=1.3)
        ax.set_xlim(lo, hi)
        ax.set_ylim(lo, hi)
        ax.set_title(name, fontsize=17)
        ax.set_xlabel("AWE T (K)")
        if ax is axs[0]:
            ax.set_ylabel("SABER T (K)")
        ax.grid(True, linestyle=":", linewidth=0.6)

    fig.suptitle("AWE vs SABER Temperature — Methods 1–4", fontsize=20, y=1.1)
    fig.savefig(out_dir / "scatter_all_methods.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

def make_combined_hist(dT_dict, out_dir):
    methods = ["Method 1", "Method 2", "Method 3", "Method 4"]
    # Common bin edges for all panels
    bins = np.arange(DELTA_T_MIN, DELTA_T_MAX + 1, 1.0)

    # Precompute hist counts to find a common ymax
    max_count = 0
    for name in methods:
        dT = dT_dict[name]
        clipped = np.clip(dT, DELTA_T_MIN, DELTA_T_MAX)
        counts, _ = np.histogram(clipped, bins=bins)
        max_count = max(max_count, counts.max())

    # Add a little headroom
    ymax = int(np.ceil(max_count * 1.10))

    # Build the shared-y subplot grid
    fig, axs = plt.subplots(1, 4, figsize=(17, 4.5), constrained_layout=True, sharey=True)
    for ax, name in zip(axs, methods):
        dT = dT_dict[name]
        ax.hist(np.clip(dT, DELTA_T_MIN, DELTA_T_MAX), bins=bins,
                edgecolor="black", linewidth=0.8, alpha=0.85)
        ax.axvline(0, color="k", linestyle="--", linewidth=1.2)
        ax.set_xlim(DELTA_T_MIN, DELTA_T_MAX)
        ax.set_ylim(0, ymax)
        ax.set_title(name, fontsize=17)
        ax.set_xlabel("ΔT (K)")
        if ax is axs[0]:
            ax.set_ylabel("Count")
        ax.grid(axis="y", linestyle="--", linewidth=0.5)

    fig.suptitle("ΔT Histograms — Methods 1–4", fontsize=20, y=1.1)
    fig.savefig(out_dir / "hist_all_methods.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

# --------------------- main ---------------------
def run_all_methods():
    csv_path_123 = Path(CSV_PATH_123)
    csv_path_m4  = Path(CSV_PATH_M4)
    df123 = pd.read_csv(csv_path_123)
    dfm4  = pd.read_csv(csv_path_m4)

    out_dir = csv_path_123.parent / "all_methods_plots_large"
    out_dir.mkdir(exist_ok=True, parents=True)

    awe_saber_dict, dT_dict = {}, {}
    for method_name, (which_csv, saber_col, delta_col, tag) in METHODS.items():
        df_raw = df123 if which_csv == "123" else dfm4
        df = clean_df(df_raw, [saber_col, delta_col])
        print(f"Processing {method_name}...")
        aweT, saberT, dT = method_stats_and_plots(df, method_name, saber_col, delta_col, tag, out_dir)
        awe_saber_dict[method_name] = (aweT, saberT)
        dT_dict[method_name] = dT

    make_combined_scatter(awe_saber_dict, out_dir)
    make_combined_hist(dT_dict, out_dir)
    print(f"\n✅ All plots saved to: {out_dir}")

# Run
run_all_methods()


Processing Method 1...
Processing Method 2...
Processing Method 3...
Processing Method 4...

✅ All plots saved to: all_methods_plots_large


In [17]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Summary + plots for SABER Methods 1–4 (paper version, with trend lines)
All plots saved to one folder, larger text for visibility on paper.
"""

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from datetime import datetime
import warnings

# --------------------- user settings ---------------------
CSV_PATH_123 = "matched_all_2024_2025_methods_ratio90_clean.csv"
CSV_PATH_M4  = "matched_all_2024_2025_m4shifts_combined_ratio90_clean.csv"

# Common columns
COL_AWE_T = "awe_temp_avg_unfiltered"
COL_LAT   = "saber_lat"
COL_LON   = "saber_lon"
COL_TIME  = "saber_time"

# Plot ranges
DELTA_T_MIN, DELTA_T_MAX = -25.0, 25.0
LAT_MIN, LAT_MAX = -60.0, 60.0

# Methods
METHODS = {
    "Method 1": ("123", "saber_temp",                    "delta_temp_simple_unfiltered_K",           "m1"),
    "Method 2": ("123", "m2_T_K",                        "delta_temp_m2_unfiltered_K",               "m2"),
    "Method 3": ("123", "m3_T_K",                        "delta_temp_m3_unfiltered_K",               "m3"),
    "Method 4": ("m4",  "m4_T_K_shift_m0p75",            "delta_temp_m4_unfiltered_K_shift_m0p75",   "m4_m0p75"),
}

# --------------------- style for paper ---------------------
plt.rcParams.update({
    "font.size": 19,
    "axes.labelsize": 20,
    "axes.titlesize": 21,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 18,
    "figure.titlesize": 23,
    "lines.linewidth": 1.8,
})

# --------------------- helpers ---------------------
def parse_iso_time_to_datetime(s):
    try:
        return datetime.fromisoformat(str(s))
    except Exception:
        try:
            return pd.to_datetime(s).to_pydatetime()
        except Exception:
            return None

def compute_doy_across_2024_2025(dt):
    if dt is None:
        return np.nan
    if dt.year == 2024:
        return float(dt.timetuple().tm_yday)
    if dt.year == 2025:
        return float(dt.timetuple().tm_yday + 366)
    base = datetime(2024, 1, 1)
    return float((dt - base).days + 1)

def clean_df(df, needed_cols):
    for col in needed_cols + [COL_AWE_T, COL_LAT, COL_LON, COL_TIME]:
        if col not in df.columns:
            raise KeyError(f"Missing column: {col}")
    df = df.copy()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=FutureWarning)
        dt_list = [parse_iso_time_to_datetime(x) for x in df[COL_TIME].values]
    df["__dt"]  = dt_list
    df["__doy"] = [compute_doy_across_2024_2025(d) for d in dt_list]
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=[COL_AWE_T, COL_LAT, COL_LON, "__doy"] + needed_cols)
    return df

def iqr(x):
    q75 = np.nanpercentile(x, 75)
    q25 = np.nanpercentile(x, 25)
    return q75 - q25

def within(x, thr):
    x = np.asarray(x, dtype=float)
    return np.mean(np.abs(x) <= thr) * 100.0

# --------------------- main plotting helper ---------------------
def method_stats_and_plots(df, method_name, saber_col, delta_col, tag, out_dir):
    aweT   = df[COL_AWE_T].values
    saberT = df[saber_col].values
    dT     = df[delta_col].values
    lat    = df[COL_LAT].values
    lon    = df[COL_LON].values
    doy    = df["__doy"].values

    # Statistics (not all used yet but may be handy)
    n_total = len(df)
    mean_dt = np.nanmean(dT)
    std_dt  = np.nanstd(dT, ddof=1)
    med_dt  = np.nanmedian(dT)
    iqr_dt  = iqr(dT)
    rmse_dt = np.sqrt(np.nanmean((dT)**2))
    mae_dt  = np.nanmean(np.abs(dT))

    # Corr + fit (mask NaNs to avoid issues)
    mask_fit = np.isfinite(aweT) & np.isfinite(saberT)
    if np.sum(mask_fit) >= 2:
        slope, intercept = np.polyfit(aweT[mask_fit], saberT[mask_fit], 1)
        r = np.corrcoef(aweT[mask_fit], saberT[mask_fit])[0, 1]
    else:
        slope, intercept, r = np.nan, np.nan, np.nan

    # ----- per-method plots -----
    # 1) Scatter AWE vs SABER with 1:1, trend line, and equation text
    plt.figure(figsize=(7.2, 7.2))
    plt.scatter(aweT, saberT, s=10, alpha=0.55, label="Data")

    lo = min(np.nanmin(aweT), np.nanmin(saberT)) - 2
    hi = max(np.nanmax(aweT), np.nanmax(saberT)) + 2

    # 1:1 line
    plt.plot([lo, hi], [lo, hi], "--", color="gray", linewidth=1.3, label="1:1 line")

    # Trend line
    if np.isfinite(slope) and np.isfinite(intercept):
        x_line = np.linspace(lo, hi, 200)
        y_line = slope * x_line + intercept
        plt.plot(x_line, y_line, color="tab:red", linewidth=1.7, label="Trend line")

        # Display equation + r
        eq_text = (
            f"y = {slope:.3f} x + {intercept:.3f}\n"
            f"r = {r:.3f}"
        )
        plt.text(
            0.05, 0.95, eq_text,
            transform=plt.gca().transAxes,
            fontsize=16,
            verticalalignment="top",
            bbox=dict(facecolor="white", alpha=0.70, edgecolor="black")
        )

    plt.xlabel("AWE T (K)")
    plt.ylabel(f"SABER T (K)\n{method_name}")
    plt.xlim(lo, hi)
    plt.ylim(lo, hi)
    plt.grid(True, linestyle="--", alpha=0.5)

    # Legend moved to lower-right so it doesn't fight the equation box
    plt.legend(loc="lower right", fontsize=14, framealpha=0.7)

    plt.tight_layout()
    plt.savefig(out_dir / f"scatter_awe_vs_saber_{tag}.png", dpi=300)
    plt.close()

    # 2) Histogram
    plt.figure(figsize=(8, 5))
    bins = np.arange(DELTA_T_MIN, DELTA_T_MAX + 1, 1.0)
    plt.hist(np.clip(dT, DELTA_T_MIN, DELTA_T_MAX),
             bins=bins, edgecolor="black", linewidth=0.8, alpha=0.85)
    plt.axvline(0, color="k", linestyle="--", linewidth=1.2)
    plt.xlabel("ΔT (SABER - AWE) [K]")
    plt.ylabel("Count")
    plt.title(f"ΔT Distribution — {method_name}")
    plt.grid(axis="y", linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.savefig(out_dir / f"hist_deltaT_{tag}.png", dpi=300)
    plt.close()

    # 3) Map (optional Basemap)
    try:
        from mpl_toolkits.basemap import Basemap
        sel = (lat >= LAT_MIN) & (lat <= LAT_MAX) & np.isfinite(dT)
        plt.figure(figsize=(12, 5.2))
        m = Basemap(projection="cyl",
                    llcrnrlat=LAT_MIN, urcrnrlat=LAT_MAX,
                    llcrnrlon=-180, urcrnrlon=180, resolution="c")
        m.drawcoastlines(linewidth=0.6)
        m.drawparallels(np.arange(-60, 61, 15), labels=[1,0,0,0], fontsize=13)
        m.drawmeridians(np.arange(-180, 181, 30), labels=[0,0,0,1], fontsize=13)
        norm = TwoSlopeNorm(vmin=DELTA_T_MIN, vcenter=0, vmax=DELTA_T_MAX)
        sc = m.scatter(lon[sel], lat[sel], c=dT[sel], s=8, cmap="coolwarm", norm=norm, alpha=0.85)
        cb = plt.colorbar(sc, shrink=0.85, pad=0.03)
        cb.set_label("ΔT (K)", fontsize=15)
        plt.title(f"ΔT Map (lat -60..60) — {method_name}", fontsize=18)
        plt.tight_layout()
        plt.savefig(out_dir / f"map_deltaT_{tag}_basemap_rect.png", dpi=300)
        plt.close()
    except Exception:
        pass

    # 4) DOY vs Latitude
    sel2 = (lat >= LAT_MIN) & (lat <= LAT_MAX) & np.isfinite(dT)
    plt.figure(figsize=(11, 5.5))
    norm = TwoSlopeNorm(vmin=DELTA_T_MIN, vcenter=0.0, vmax=DELTA_T_MAX)
    sc = plt.scatter(doy[sel2], lat[sel2], c=dT[sel2], s=8, cmap="coolwarm", norm=norm, alpha=0.85)
    plt.xlabel("Day since 2024-01-01")
    plt.ylabel("Latitude (°)")
    plt.colorbar(sc, pad=0.015, label="ΔT (K)")
    plt.title(f"ΔT vs DOY & Latitude — {method_name}", fontsize=18)
    plt.tight_layout()
    plt.savefig(out_dir / f"doy_lat_deltaT_{tag}.png", dpi=300)
    plt.close()

    # Return for combined plots
    return aweT, saberT, dT, slope, intercept


# --------------------- combined 1×4 plots ---------------------
def make_combined_scatter(awe_saber_dict, fit_dict, out_dir):
    methods = ["Method 1", "Method 2", "Method 3", "Method 4"]

    fig, axs = plt.subplots(
        1, 4, figsize=(17, 4.5),
        constrained_layout=True, sharex=True, sharey=True
    )

    # Determine global min/max across all methods
    mins, maxs = [], []
    for name in methods:
        aweT, saberT = awe_saber_dict[name]
        mins.append(np.nanmin([aweT, saberT]))
        maxs.append(np.nanmax([aweT, saberT]))
    lo, hi = float(min(mins)) - 2, float(max(maxs)) + 2

    # Handles for shared legend
    legend_handles = []
    legend_labels  = []

    for ax, name in zip(axs, methods):
        aweT, saberT = awe_saber_dict[name]
        slope, intercept = fit_dict[name]

        # Scatter
        h_sc = ax.scatter(aweT, saberT, s=10, alpha=0.55)
        if not legend_handles:
            legend_handles.append(h_sc)
            legend_labels.append("Data")

        # 1:1 line
        h_11 = ax.plot([lo, hi], [lo, hi], "--", color="gray", linewidth=1.3)[0]
        if len(legend_handles) == 1:
            legend_handles.append(h_11)
            legend_labels.append("1:1 line")

        # Trend line
        if np.isfinite(slope) and np.isfinite(intercept):
            x_line = np.linspace(lo, hi, 200)
            y_line = slope * x_line + intercept
            h_tr = ax.plot(x_line, y_line, color="tab:red", linewidth=1.6)[0]

            if len(legend_handles) == 2:
                legend_handles.append(h_tr)
                legend_labels.append("Trend line")

            # Equation text inside panel
            eq_text = f"y = {slope:.3f} x + {intercept:.3f}"
            ax.text(
                0.04, 0.95, eq_text,
                transform=ax.transAxes,
                fontsize=13,
                verticalalignment="top",
                bbox=dict(facecolor="white", alpha=0.55, edgecolor="black")
            )

        ax.set_xlim(lo, hi)
        ax.set_ylim(lo, hi)
        ax.set_title(name, fontsize=17)
        ax.set_xlabel("AWE T (K)")
        if ax is axs[0]:
            ax.set_ylabel("SABER T (K)")
        ax.grid(True, linestyle=":", linewidth=0.6)

    # Shared legend bottom-right of the whole figure
    fig.legend(
        legend_handles, legend_labels,
        loc="lower right",
        bbox_to_anchor=(0.986, 0.16),
        fontsize=15,
        framealpha=0.8
    )

    fig.suptitle("AWE vs SABER Temperature — Methods 1–4", fontsize=20, y=1.05)
    fig.savefig(out_dir / "scatter_all_methods.png", dpi=300, bbox_inches="tight")
    plt.close(fig)


def make_combined_hist(dT_dict, out_dir):
    methods = ["Method 1", "Method 2", "Method 3", "Method 4"]
    bins = np.arange(DELTA_T_MIN, DELTA_T_MAX + 1, 1.0)

    # Precompute hist counts to find a common ymax
    max_count = 0
    for name in methods:
        dT = dT_dict[name]
        clipped = np.clip(dT, DELTA_T_MIN, DELTA_T_MAX)
        counts, _ = np.histogram(clipped, bins=bins)
        max_count = max(max_count, counts.max())

    ymax = int(np.ceil(max_count * 1.10))

    fig, axs = plt.subplots(1, 4, figsize=(17, 4.5), constrained_layout=True, sharey=True)
    for ax, name in zip(axs, methods):
        dT = dT_dict[name]
        ax.hist(np.clip(dT, DELTA_T_MIN, DELTA_T_MAX), bins=bins,
                edgecolor="black", linewidth=0.8, alpha=0.85)
        ax.axvline(0, color="k", linestyle="--", linewidth=1.2)
        ax.set_xlim(DELTA_T_MIN, DELTA_T_MAX)
        ax.set_ylim(0, ymax)
        ax.set_title(name, fontsize=17)
        ax.set_xlabel("ΔT (K)")
        if ax is axs[0]:
            ax.set_ylabel("Count")
        ax.grid(axis="y", linestyle="--", linewidth=0.5)

    fig.suptitle("ΔT Histograms — Methods 1–4", fontsize=20, y=1.05)
    fig.savefig(out_dir / "hist_all_methods.png", dpi=300, bbox_inches="tight")
    plt.close(fig)


# --------------------- main ---------------------
def run_all_methods():
    csv_path_123 = Path(CSV_PATH_123)
    csv_path_m4  = Path(CSV_PATH_M4)
    df123 = pd.read_csv(csv_path_123)
    dfm4  = pd.read_csv(csv_path_m4)

    out_dir = csv_path_123.parent / "all_methods_plots_large"
    out_dir.mkdir(exist_ok=True, parents=True)

    awe_saber_dict, dT_dict, fit_dict = {}, {}, {}
    for method_name, (which_csv, saber_col, delta_col, tag) in METHODS.items():
        df_raw = df123 if which_csv == "123" else dfm4
        df = clean_df(df_raw, [saber_col, delta_col])
        print(f"Processing {method_name}...")
        aweT, saberT, dT, slope, intercept = method_stats_and_plots(
            df, method_name, saber_col, delta_col, tag, out_dir
        )
        awe_saber_dict[method_name] = (aweT, saberT)
        dT_dict[method_name] = dT
        fit_dict[method_name] = (slope, intercept)

    make_combined_scatter(awe_saber_dict, fit_dict, out_dir)
    make_combined_hist(dT_dict, out_dir)
    print(f"\n✅ All plots saved to: {out_dir}")


# Run
if __name__ == "__main__":
    run_all_methods()


Processing Method 1...
Processing Method 2...
Processing Method 3...
Processing Method 4...

✅ All plots saved to: all_methods_plots_large


Should we use symmetric (orthogonal) regression instead?

Possibly — if:

Both AWE and SABER include uncertainty

Neither is a true independent variable

We want the line that best represents the relationship rather than predicting SABER from AWE

But the standard OLS we’re using is completely fine for most comparisons.

In [25]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Summary + plots for SABER Methods 1–4 (paper version, with OLS and orthogonal fits)

- OLS fit: SABER vs AWE (minimize vertical distance dy)
- Orthogonal fit (Total Least Squares): minimize perpendicular distance
All plots saved to one folder, larger text for visibility on paper.
"""

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from datetime import datetime
import warnings

# --------------------- user settings ---------------------
CSV_PATH_123 = "matched_all_2024_2025_methods_ratio90_clean.csv"
CSV_PATH_M4  = "matched_all_2024_2025_m4shifts_combined_ratio90_clean.csv"

# Common columns
COL_AWE_T = "awe_temp_avg_unfiltered"
COL_LAT   = "saber_lat"
COL_LON   = "saber_lon"
COL_TIME  = "saber_time"

# Plot ranges
DELTA_T_MIN, DELTA_T_MAX = -25.0, 25.0
LAT_MIN, LAT_MAX = -60.0, 60.0

# Methods
METHODS = {
    "Method 1": ("123", "saber_temp",                    "delta_temp_simple_unfiltered_K",           "m1"),
    "Method 2": ("123", "m2_T_K",                        "delta_temp_m2_unfiltered_K",               "m2"),
    "Method 3": ("123", "m3_T_K",                        "delta_temp_m3_unfiltered_K",               "m3"),
    "Method 4": ("m4",  "m4_T_K_shift_m0p75",            "delta_temp_m4_unfiltered_K_shift_m0p75",   "m4_m0p75"),
}

# --------------------- style for paper ---------------------
plt.rcParams.update({
    "font.size": 19,
    "axes.labelsize": 20,
    "axes.titlesize": 21,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 18,
    "figure.titlesize": 23,
    "lines.linewidth": 1.8,
})


# --------------------- helpers ---------------------
def parse_iso_time_to_datetime(s):
    try:
        return datetime.fromisoformat(str(s))
    except Exception:
        try:
            return pd.to_datetime(s).to_pydatetime()
        except Exception:
            return None

def compute_doy_across_2024_2025(dt):
    if dt is None:
        return np.nan
    if dt.year == 2024:
        return float(dt.timetuple().tm_yday)
    if dt.year == 2025:
        return float(dt.timetuple().tm_yday + 366)
    base = datetime(2024, 1, 1)
    return float((dt - base).days + 1)

def clean_df(df, needed_cols):
    for col in needed_cols + [COL_AWE_T, COL_LAT, COL_LON, COL_TIME]:
        if col not in df.columns:
            raise KeyError(f"Missing column: {col}")
    df = df.copy()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=FutureWarning)
        dt_list = [parse_iso_time_to_datetime(x) for x in df[COL_TIME].values]
    df["__dt"]  = dt_list
    df["__doy"] = [compute_doy_across_2024_2025(d) for d in dt_list]
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=[COL_AWE_T, COL_LAT, COL_LON, "__doy"] + needed_cols)
    return df

def iqr(x):
    q75 = np.nanpercentile(x, 75)
    q25 = np.nanpercentile(x, 25)
    return q75 - q25

def within(x, thr):
    x = np.asarray(x, dtype=float)
    return np.mean(np.abs(x) <= thr) * 100.0

def orthogonal_fit(x, y):
    """
    Total Least Squares / Orthogonal regression line y = m x + b
    minimizing perpendicular distance.

    Returns slope, intercept. If it fails, returns (nan, nan).
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if np.sum(mask) < 2:
        return np.nan, np.nan

    x_fit = x[mask]
    y_fit = y[mask]

    x_mean = x_fit.mean()
    y_mean = y_fit.mean()

    # Center data
    X = np.vstack((x_fit - x_mean, y_fit - y_mean)).T  # shape (N, 2)

    # SVD
    try:
        _, _, vt = np.linalg.svd(X, full_matrices=False)
    except np.linalg.LinAlgError:
        return np.nan, np.nan

    # The direction of smallest variance (normal to line)
    a, b = vt[-1]  # normal vector (a, b) so that a*(x-x_mean) + b*(y-y_mean) = 0

    if np.abs(b) < 1e-12:
        # Vertical line case; cannot express as y = m x + c
        return np.nan, np.nan

    slope = -a / b
    intercept = y_mean - slope * x_mean
    return slope, intercept


# --------------------- main plotting helper ---------------------
def method_stats_and_plots(df, method_name, saber_col, delta_col, tag, out_dir):
    aweT   = df[COL_AWE_T].values
    saberT = df[saber_col].values
    dT     = df[delta_col].values
    lat    = df[COL_LAT].values
    lon    = df[COL_LON].values
    doy    = df["__doy"].values

    # Some stats (unused for plots but handy)
    n_total = len(df)
    mean_dt = np.nanmean(dT)
    std_dt  = np.nanstd(dT, ddof=1)
    med_dt  = np.nanmedian(dT)
    iqr_dt  = iqr(dT)
    rmse_dt = np.sqrt(np.nanmean((dT)**2))
    mae_dt  = np.nanmean(np.abs(dT))

    # Corr + OLS fit
    mask_fit = np.isfinite(aweT) & np.isfinite(saberT)
    if np.sum(mask_fit) >= 2:
        slope_ols, intercept_ols = np.polyfit(aweT[mask_fit], saberT[mask_fit], 1)
        r = np.corrcoef(aweT[mask_fit], saberT[mask_fit])[0, 1]
    else:
        slope_ols, intercept_ols, r = np.nan, np.nan, np.nan

    # Orthogonal / TLS fit
    slope_tls, intercept_tls = orthogonal_fit(aweT, saberT)

    # ----- per-method plots -----
    # 1) Scatter AWE vs SABER with 1:1, OLS, TLS, equations
    plt.figure(figsize=(7.2, 7.2))
    plt.scatter(aweT, saberT, s=10, alpha=0.55, label="Data")

    lo = min(np.nanmin(aweT), np.nanmin(saberT)) - 2
    hi = max(np.nanmax(aweT), np.nanmax(saberT)) + 2

    # 1:1 line
    plt.plot([lo, hi], [lo, hi], "--", color="gray", linewidth=1.3, label="1:1 line")

    x_line = np.linspace(lo, hi, 200)

    # OLS fit line
    if np.isfinite(slope_ols) and np.isfinite(intercept_ols):
        y_ols = slope_ols * x_line + intercept_ols
        plt.plot(x_line, y_ols, color="tab:red", linewidth=1.8, label="OLS fit")

    # TLS (orthogonal) fit line
    if np.isfinite(slope_tls) and np.isfinite(intercept_tls):
        y_tls = slope_tls * x_line + intercept_tls
        plt.plot(x_line, y_tls, color="tab:green", linestyle="--", linewidth=1.8, label="Orthogonal fit")

    # Equation text
    eq_lines = []
    if np.isfinite(slope_ols) and np.isfinite(intercept_ols):
        eq_lines.append(f"OLS:  y = {slope_ols:.3f} x + {intercept_ols:.3f}")
    if np.isfinite(slope_tls) and np.isfinite(intercept_tls):
        eq_lines.append(f"Orth: y = {slope_tls:.3f} x + {intercept_tls:.3f}")
    if np.isfinite(r):
        eq_lines.append(f"r = {r:.3f}")

    if eq_lines:
        eq_text = "\n".join(eq_lines)
        plt.text(
            0.05, 0.95, eq_text,
            transform=plt.gca().transAxes,
            fontsize=15,
            verticalalignment="top",
            bbox=dict(facecolor="white", alpha=0.75, edgecolor="black")
        )

    plt.xlabel("AWE T (K)")
    plt.ylabel(f"SABER T (K)\n{method_name}")
    plt.xlim(lo, hi)
    plt.ylim(lo, hi)
    plt.grid(True, linestyle="--", alpha=0.5)

    # Legend in lower-right to avoid equation
    plt.legend(loc="lower right", fontsize=14, framealpha=0.7)
    plt.tight_layout()
    plt.savefig(out_dir / f"scatter_awe_vs_saber_{tag}.png", dpi=300)
    plt.close()

    # 2) Histogram
    plt.figure(figsize=(8, 5))
    bins = np.arange(DELTA_T_MIN, DELTA_T_MAX + 1, 1.0)
    plt.hist(
        np.clip(dT, DELTA_T_MIN, DELTA_T_MAX),
        bins=bins, edgecolor="black", linewidth=0.8, alpha=0.85
    )
    plt.axvline(0, color="k", linestyle="--", linewidth=1.2)
    plt.xlabel("ΔT (SABER - AWE) [K]")
    plt.ylabel("Count")
    plt.title(f"ΔT Distribution — {method_name}")
    plt.grid(axis="y", linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.savefig(out_dir / f"hist_deltaT_{tag}.png", dpi=300)
    plt.close()

    # 3) Map (optional Basemap)
    try:
        from mpl_toolkits.basemap import Basemap
        sel = (lat >= LAT_MIN) & (lat <= LAT_MAX) & np.isfinite(dT)
        plt.figure(figsize=(12, 5.2))
        m = Basemap(
            projection="cyl",
            llcrnrlat=LAT_MIN, urcrnrlat=LAT_MAX,
            llcrnrlon=-180, urcrnrlon=180, resolution="c"
        )
        m.drawcoastlines(linewidth=0.6)
        m.drawparallels(np.arange(-60, 61, 15), labels=[1, 0, 0, 0], fontsize=13)
        m.drawmeridians(np.arange(-180, 181, 30), labels=[0, 0, 0, 1], fontsize=13)
        norm = TwoSlopeNorm(vmin=DELTA_T_MIN, vcenter=0, vmax=DELTA_T_MAX)
        sc = m.scatter(
            lon[sel], lat[sel], c=dT[sel], s=8,
            cmap="coolwarm", norm=norm, alpha=0.85
        )
        cb = plt.colorbar(sc, shrink=0.85, pad=0.03)
        cb.set_label("ΔT (K)", fontsize=15)
        plt.title(f"ΔT Map (lat -60..60) — {method_name}", fontsize=18)
        plt.tight_layout()
        plt.savefig(out_dir / f"map_deltaT_{tag}_basemap_rect.png", dpi=300)
        plt.close()
    except Exception:
        pass

    # 4) DOY vs Latitude
    sel2 = (lat >= LAT_MIN) & (lat <= LAT_MAX) & np.isfinite(dT)
    plt.figure(figsize=(11, 5.5))
    norm = TwoSlopeNorm(vmin=DELTA_T_MIN, vcenter=0.0, vmax=DELTA_T_MAX)
    sc = plt.scatter(
        doy[sel2], lat[sel2],
        c=dT[sel2], s=8, cmap="coolwarm", norm=norm, alpha=0.85
    )
    plt.xlabel("Day since 2024-01-01")
    plt.ylabel("Latitude (°)")
    plt.colorbar(sc, pad=0.015, label="ΔT (K)")
    plt.title(f"ΔT vs DOY & Latitude — {method_name}", fontsize=18)
    plt.tight_layout()
    plt.savefig(out_dir / f"doy_lat_deltaT_{tag}.png", dpi=300)
    plt.close()

    # Return for combined plots
    return aweT, saberT, dT, slope_ols, intercept_ols, slope_tls, intercept_tls


# --------------------- combined 1×4 plots ---------------------
def make_combined_scatter(awe_saber_dict, fit_dict_ols, fit_dict_tls, out_dir):
    methods = ["Method 1", "Method 2", "Method 3", "Method 4"]

    fig, axs = plt.subplots(
        1, 4, figsize=(17, 4.5),
        constrained_layout=True, sharex=True, sharey=True
    )

    # Determine global min/max
    mins, maxs = [], []
    for name in methods:
        aweT, saberT = awe_saber_dict[name]
        mins.append(np.nanmin([aweT, saberT]))
        maxs.append(np.nanmax([aweT, saberT]))
    lo, hi = float(min(mins)) - 2, float(max(maxs)) + 2

    legend_handles = []
    legend_labels  = []

    for ax, name in zip(axs, methods):
        aweT, saberT = awe_saber_dict[name]
        slope_ols, intercept_ols = fit_dict_ols[name]
        slope_tls, intercept_tls = fit_dict_tls[name]

        # Scatter
        h_sc = ax.scatter(aweT, saberT, s=10, alpha=0.55)
        if not legend_handles:
            legend_handles.append(h_sc)
            legend_labels.append("Data")

        # 1:1 line
        h_11 = ax.plot([lo, hi], [lo, hi], "--", color="gray", linewidth=1.3)[0]
        if len(legend_handles) == 1:
            legend_handles.append(h_11)
            legend_labels.append("1:1 line")

        x_line = np.linspace(lo, hi, 200)

        # OLS fit
        if np.isfinite(slope_ols) and np.isfinite(intercept_ols):
            y_ols = slope_ols * x_line + intercept_ols
            h_ols = ax.plot(x_line, y_ols, color="tab:red", linewidth=1.6)[0]
            if len(legend_handles) == 2:
                legend_handles.append(h_ols)
                legend_labels.append("OLS fit")

        # TLS fit
        if np.isfinite(slope_tls) and np.isfinite(intercept_tls):
            y_tls = slope_tls * x_line + intercept_tls
            h_tls = ax.plot(x_line, y_tls, color="tab:green", linestyle="--", linewidth=1.6)[0]
            if len(legend_handles) == 3:
                legend_handles.append(h_tls)
                legend_labels.append("Orthogonal fit")

        # Show just OLS equation inside panel to keep it readable
        if np.isfinite(slope_ols) and np.isfinite(intercept_ols):
            eq_text = f"OLS: y = {slope_ols:.3f} x + {intercept_ols:.3f}"
            ax.text(
                0.04, 0.95, eq_text,
                transform=ax.transAxes,
                fontsize=12,
                verticalalignment="top",
                bbox=dict(facecolor="white", alpha=0.55, edgecolor="black")
            )

        ax.set_xlim(lo, hi)
        ax.set_ylim(lo, hi)
        ax.set_title(name, fontsize=17)
        ax.set_xlabel("AWE T (K)")
        if ax is axs[0]:
            ax.set_ylabel("SABER T (K)")
        ax.grid(True, linestyle=":", linewidth=0.6)

    # Shared legend at bottom-right of the whole figure
    fig.legend(
        legend_handles, legend_labels,
        loc="lower right",
        bbox_to_anchor=(0.987, 0.15),
        fontsize=15,
        framealpha=0.8
    )

    fig.suptitle("AWE vs SABER Temperature — Methods 1–4", fontsize=20, y=1.05)
    fig.savefig(out_dir / "scatter_all_methods.png", dpi=300, bbox_inches="tight")
    plt.close(fig)


def make_combined_hist(dT_dict, out_dir):
    methods = ["Method 1", "Method 2", "Method 3", "Method 4"]
    bins = np.arange(DELTA_T_MIN, DELTA_T_MAX + 1, 1.0)

    max_count = 0
    for name in methods:
        dT = dT_dict[name]
        clipped = np.clip(dT, DELTA_T_MIN, DELTA_T_MAX)
        counts, _ = np.histogram(clipped, bins=bins)
        max_count = max(max_count, counts.max())

    ymax = int(np.ceil(max_count * 1.10))

    fig, axs = plt.subplots(1, 4, figsize=(17, 4.5), constrained_layout=True, sharey=True)
    for ax, name in zip(axs, methods):
        dT = dT_dict[name]
        ax.hist(
            np.clip(dT, DELTA_T_MIN, DELTA_T_MAX), bins=bins,
            edgecolor="black", linewidth=0.8, alpha=0.85
        )
        ax.axvline(0, color="k", linestyle="--", linewidth=1.2)
        ax.set_xlim(DELTA_T_MIN, DELTA_T_MAX)
        ax.set_ylim(0, ymax)
        ax.set_title(name, fontsize=17)
        ax.set_xlabel("ΔT (K)")
        if ax is axs[0]:
            ax.set_ylabel("Count")
        ax.grid(axis="y", linestyle="--", linewidth=0.5)

    fig.suptitle("ΔT Histograms — Methods 1–4", fontsize=20, y=1.05)
    fig.savefig(out_dir / "hist_all_methods.png", dpi=300, bbox_inches="tight")
    plt.close(fig)


# --------------------- main ---------------------
def run_all_methods():
    csv_path_123 = Path(CSV_PATH_123)
    csv_path_m4  = Path(CSV_PATH_M4)
    df123 = pd.read_csv(csv_path_123)
    dfm4  = pd.read_csv(csv_path_m4)

    out_dir = csv_path_123.parent / "all_methods_plots_large"
    out_dir.mkdir(exist_ok=True, parents=True)

    awe_saber_dict = {}
    dT_dict = {}
    fit_dict_ols = {}
    fit_dict_tls = {}

    for method_name, (which_csv, saber_col, delta_col, tag) in METHODS.items():
        df_raw = df123 if which_csv == "123" else dfm4
        df = clean_df(df_raw, [saber_col, delta_col])
        print(f"Processing {method_name}...")
        aweT, saberT, dT, slope_ols, intercept_ols, slope_tls, intercept_tls = method_stats_and_plots(
            df, method_name, saber_col, delta_col, tag, out_dir
        )
        awe_saber_dict[method_name] = (aweT, saberT)
        dT_dict[method_name] = dT
        fit_dict_ols[method_name] = (slope_ols, intercept_ols)
        fit_dict_tls[method_name] = (slope_tls, intercept_tls)

    make_combined_scatter(awe_saber_dict, fit_dict_ols, fit_dict_tls, out_dir)
    make_combined_hist(dT_dict, out_dir)
    print(f"\n✅ All plots saved to: {out_dir}")


# Run
if __name__ == "__main__":
    run_all_methods()


Processing Method 1...
Processing Method 2...
Processing Method 3...
Processing Method 4...

✅ All plots saved to: all_methods_plots_large
